## 1. Import libraries

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from PIL import Image
from tensorflow.keras import layers, Model

tf.keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## 2. Find project files

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
# Dynamic multi-directional project root resolver
here = Path.cwd().resolve()
ROOT = None
for candidate in [here, *here.parents, here / "SIH2026-main"]:
    if candidate.is_dir() and (candidate / "dataset").is_dir() and (candidate / "ml").is_dir():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    for d in here.glob("**/ml"):
        if (d.parent / "dataset").is_dir():
            ROOT = d.parent.resolve()
            break
assert ROOT is not None, "Project root not found (must contain 'dataset' and 'ml')"

DATA = ROOT / "ml" / "processed"
MODEL_DIR = ROOT / "ml" / "models"
RESULT_DIR = ROOT / "ml" / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

config = json.load(open(DATA / "preprocess_config.json"))
df = pd.read_csv(DATA / "dataset_splits.csv")

IMG_SIZE = tuple(config["img_size"])
THRESHOLD = config["mask_threshold"]
MEAN = np.array(config["mean"], dtype=np.float32)
STD = np.array(config["std"], dtype=np.float32)

train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)

print("Project root:", ROOT)
print("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))


## 3. Load data in small batches

This loads only a few images at a time, instead of keeping the full dataset in RAM.


In [ ]:
class ImageLoader(tf.keras.utils.Sequence):
    def __init__(self, data, batch_size=2, shuffle=False):
        super().__init__()
        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.order = np.arange(len(data))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        ids = self.order[index*self.batch_size:(index+1)*self.batch_size]
        images, masks = [], []

        for i in ids:
            row = self.data.iloc[i]
            img = Image.open(ROOT / row.image_path).convert("RGB").resize(IMG_SIZE)
            img = np.array(img, dtype="float32") / 255.0
            img = (img - MEAN) / STD

            mask = Image.open(ROOT / row.mask_path).convert("L")
            mask = mask.resize(IMG_SIZE, Image.Resampling.NEAREST)
            mask = (np.array(mask) > THRESHOLD).astype("float32")

            images.append(img)
            masks.append(mask[..., None])

        return np.array(images), np.array(masks)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.order)


BATCH_SIZE = 2
train_loader = ImageLoader(train_df, BATCH_SIZE, shuffle=True)
val_loader = ImageLoader(val_df, BATCH_SIZE)
test_loader = ImageLoader(test_df, BATCH_SIZE)

x_sample, y_sample = train_loader[0]
print("Batch image shape:", x_sample.shape)
print("Batch mask shape:", y_sample.shape)


## 4. View one example

In [ ]:
plt.figure(figsize=(8, 3))
plt.subplot(1, 2, 1)
plt.imshow(np.clip(x_sample[0] * STD + MEAN, 0, 1))
plt.title("SAR Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(y_sample[0, :, :, 0], cmap="gray")
plt.title("Oil Spill Mask")
plt.axis("off")
plt.show()


## 5. Dice score and loss

In [ ]:
def dice_score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2 * intersection + 1e-6) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + 1e-6
    )

def loss_fn(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + (1 - dice_score(y_true, y_pred))


## 6. Build Attention U-Net

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    return x

def attention_gate(skip, gate, filters):
    a = layers.Conv2D(filters, 1, padding="same")(skip)
    b = layers.Conv2D(filters, 1, padding="same")(gate)
    x = layers.Activation("relu")(layers.Add()([a, b]))
    x = layers.Conv2D(1, 1, activation="sigmoid")(x)
    return layers.Multiply()([skip, x])

def build_attention_unet():
    inputs = layers.Input((IMG_SIZE[1], IMG_SIZE[0], 3))

    c1 = conv_block(inputs, 16); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, 32); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, 64); p3 = layers.MaxPooling2D()(c3)
    middle = conv_block(p3, 128)

    u3 = layers.UpSampling2D()(middle)
    s3 = attention_gate(c3, u3, 64)
    d3 = conv_block(layers.Concatenate()([u3, s3]), 64)

    u2 = layers.UpSampling2D()(d3)
    s2 = attention_gate(c2, u2, 32)
    d2 = conv_block(layers.Concatenate()([u2, s2]), 32)

    u1 = layers.UpSampling2D()(d2)
    s1 = attention_gate(c1, u1, 16)
    d1 = conv_block(layers.Concatenate()([u1, s1]), 16)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(d1)
    return Model(inputs, outputs)

model = build_attention_unet()
model.summary()


## 7. Compile the model

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=loss_fn,
    metrics=[dice_score]
)

BEST_PATH = MODEL_DIR / "best_attention_unet.keras"
FINAL_PATH = MODEL_DIR / "final_attention_unet.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        BEST_PATH, save_best_only=True,
        monitor="val_dice_score", mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_score", patience=5,
        mode="max", restore_best_weights=True
    )
]


## 8. Train the model

In [ ]:
history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)


## 9. Show training graph

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["dice_score"], label="Train Dice")
plt.plot(history.history["val_dice_score"], label="Validation Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Attention U-Net Training")
plt.legend()
plt.grid()
plt.show()


## 10. Test the model

In [ ]:
best_model = tf.keras.models.load_model(
    BEST_PATH,
    custom_objects={"loss_fn": loss_fn, "dice_score": dice_score}
)

predictions = []
actual_masks = []

for i in range(len(test_loader)):
    images, masks = test_loader[i]
    predictions.append(best_model.predict(images, verbose=0))
    actual_masks.append(masks)

pred = np.concatenate(predictions)
true_mask = np.concatenate(actual_masks)
pred_mask = (pred > 0.5).astype("uint8")
true_mask = (true_mask > 0.5).astype("uint8")

intersection = np.logical_and(true_mask, pred_mask).sum()
union = np.logical_or(true_mask, pred_mask).sum()

dice = (2 * intersection) / (true_mask.sum() + pred_mask.sum() + 1e-7)
iou = intersection / (union + 1e-7)
precision = intersection / (pred_mask.sum() + 1e-7)
recall = intersection / (true_mask.sum() + 1e-7)

print("Dice:", round(dice, 4))
print("IoU:", round(iou, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))


## 11. View predictions

In [ ]:
images, masks = test_loader[0]
n = min(3, len(images))
plt.figure(figsize=(12, 3 * n))

for i in range(n):
    plt.subplot(n, 3, i*3 + 1)
    plt.imshow(np.clip(images[i] * STD + MEAN, 0, 1))
    plt.title("SAR Image")
    plt.axis("off")

    plt.subplot(n, 3, i*3 + 2)
    plt.imshow(masks[i, :, :, 0], cmap="gray")
    plt.title("Actual Mask")
    plt.axis("off")

    plt.subplot(n, 3, i*3 + 3)
    plt.imshow(pred_mask[i, :, :, 0], cmap="gray")
    plt.title("Attention U-Net Prediction")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 12. Save model and results

In [ ]:
best_model.save(FINAL_PATH)

results = pd.DataFrame([{ 
    "Model": "Attention U-Net",
    "Dice": dice,
    "IoU": iou,
    "Precision": precision,
    "Recall": recall
}])

results.to_csv(RESULT_DIR / "attention_unet_results.csv", index=False)
print(results)
print("Model saved to:", FINAL_PATH)
